# 6.2 Counting characters — the pre-embedding baseline

6.1 squashed *columns* into two dimensions. This notebook is about the step before that:
how does a piece of *text* become a row of numbers at all? The cheapest answer is to count
— which three-character sequences a chunk of text contains — and call that count its
fingerprint. `CountVectorizer` is a lookup table, not a learned function: you choose the
granularity, it counts. No GPU, no download, seconds to fit on any machine.

It is also the number to beat. 6.3 replaces this hand-designed vector with one a neural
network learned from millions of documents, and 6.4 races the two on the same task; an
embedding earns its place only if it does better than counting trigrams.

The showcase is a real published claim. In 2020 the Swiss company OrphAnalytics analysed
every message posted by "Q" — the anonymous account behind QAnon — and concluded from the
writing style that the messages were written by **two different people**: one on 4chan in
late 2017, and a second who took over after the move to 8chan
([white paper](https://www.orphanalytics.com/en/news/whitepaper202012/OrphAnalyticsQAnon2020.pdf),
[press release](https://www.prnewswire.com/news-releases/qanon-is-two-different-people-shows-machine-learning-analysis-from-orphanalytics-301192981.html)).
Their method, stripped of the "patented technology": cut the text into equal chunks, count
character trigrams, measure how far apart the chunks are, look at the picture. Four steps,
and each one is a decision. This notebook rebuilds it in a few lines of `sklearn`, then asks
the question lesson 5 taught: what does the picture actually support?


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from goad_toolkit.config import FileConfig
from goad_toolkit.datatransforms import Filter, Pipeline, TransformBase
from goad_toolkit.filehandler import FileHandler
from goad_toolkit.visualizer import PlotSettings, ProjectionPlot
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import manhattan_distances

from wa_analyzer.data import load_own_chat
from wa_analyzer.model import TextClustering


## 6.2.1 The data: every post, three sites

All of Q's posts, collected by a volunteer project on GitHub as one JSON file. `FileHandler`
downloads it once into `~/.cache/mads-dav/raw` — the same caching 03.2 used for the
sunspots, so nothing is fetched twice. The file is JSON rather than a CSV, so the handler
only fetches; `json_normalize` flattens the nested post metadata into columns.


In [ ]:
config = FileConfig(
    data_dir=Path.home() / ".cache/mads-dav",
    filename=Path("qanon-posts.json"),
    url="https://raw.githubusercontent.com/jkingsman/JSON-QAnon/main/posts.json",
    date_column=None,
)
FileHandler(config).download()

# encoding spelled out: Windows would otherwise pick a locale-specific default
with (config.data_dir / "raw" / config.filename).open(encoding="utf-8") as f:
    posts = pd.json_normalize(json.load(f)["posts"], sep="_")

posts["time"] = pd.to_datetime(posts["post_metadata_time"], unit="s")
posts["site"] = posts["post_metadata_source_site"]
posts = posts.sort_values("time").reset_index(drop=True)

print(f"{len(posts):,} posts, median {posts.text.fillna('').str.len().median():.0f} characters each")
posts.groupby("site").time.agg(["min", "max", "size"])


One row is one post, and a post is short: half of them are under a hundred characters — a
"Q" signature, a link, a one-line reply. The three sites are not three parallel sources but
three **periods**: 4chan for the first two months, 8chan until it went offline in 2019, 8kun
after. Keep that table in mind. Every label in this notebook's picture is a site, and a site
is also a date.

## 6.2.2 Cleaning as steps, and what one row becomes

Two decisions before any counting, each a pipeline step so it can be printed and argued
with. `NormaliseText` — written here, the same shape as lesson 1's steps — lowercases and
collapses whitespace, and nothing more on this first pass: it counts what was posted, links
and all, and §6.2.5 asks whether that was wise. `Filter` then drops posts under fifty
characters, because a lone "Q" or a bare link is not a writing sample.


In [ ]:
class NormaliseText(TransformBase):
    """Lowercase and collapse whitespace; optionally strip URLs, or keep only letters and spaces.

    Also records the cleaned length as `n_chars`, which the chunking below works from.
    """

    def transform(self, data: pd.DataFrame, column: str, feature: str = "text",
                  strip_urls: bool = False, letters_only: bool = False) -> pd.DataFrame:
        text = data[column].fillna("").astype(str).str.lower()
        if strip_urls:
            text = text.str.replace(r"https?://\S+", " ", regex=True)
        if letters_only:
            text = text.str.replace(r"[^a-z ]+", " ", regex=True)
        text = text.str.replace(r"\s+", " ", regex=True).str.strip()
        data[feature] = text
        data["n_chars"] = text.str.len()
        return data


cleaning = Pipeline().add(NormaliseText, column="text").add(Filter, expr="n_chars > 50")
clean = cleaning.apply(posts)
print(cleaning)
print(f"{len(clean):,} posts kept, {clean.n_chars.sum():,} characters in total")


### Chunks: the row the method needs

The paper's unit is not the post but a chunk of about 7,500 characters, and the reason is
arithmetic. A trigram count on a hundred-character post is almost all zeros, and two short
posts differ by whatever happened to be in them; a count over thousands of characters
settles into frequencies that are about the writer. The chunks are also all the **same**
size, and that matters for the distance in a moment: a longer chunk has bigger counts, so
unequal chunks would look far apart for no reason but their length.

`ChunkByCharacters` numbers consecutive posts so that each chunk holds about the same
number of characters — a hundred chunks of everything that survived cleaning — and a
`groupby` joins the text and keeps the site most of the chunk's posts came from.


In [ ]:
class ChunkByCharacters(TransformBase):
    """Number consecutive rows so that every chunk holds about `size` characters of `column`.

    With `by`, the numbering restarts for every group — one run of chunks per author, say.
    """

    def transform(self, data: pd.DataFrame, column: str, size: int, feature: str = "chunk",
                  by: str | None = None) -> pd.DataFrame:
        lengths = data[column].str.len()
        running = lengths.groupby(data[by]).cumsum() if by else lengths.cumsum()
        data[feature] = running // size
        return data


def build_chunks(frame: pd.DataFrame, size: int, label: str, by: str | None = None) -> pd.DataFrame:
    """Join consecutive rows into chunks of about `size` characters, each labelled with the
    `label` value most of its rows carry. The remainder at the end is dropped."""
    keys = [by, "chunk"] if by else ["chunk"]
    numbered = Pipeline().add(ChunkByCharacters, column="text", size=size, by=by).apply(frame)
    chunks = numbered.groupby(keys).agg(
        text=("text", " ".join),
        label=(label, lambda s: s.mode().iloc[0]),
        n_posts=("text", "size"),
    ).reset_index()
    return chunks[chunks.text.str.len() > size / 2].reset_index(drop=True)


N_CHUNKS = 100
chunk_size = clean.n_chars.sum() // N_CHUNKS
chunks = build_chunks(clean, size=chunk_size, label="site")

print(f"{len(chunks)} chunks of about {chunk_size:,} characters, {chunks.n_posts.median():.0f} posts each")
print(chunks.label.value_counts().to_string())


## 6.2.3 From text to numbers: character trigrams

A trigram is any run of three consecutive characters, spaces included.
`CountVectorizer(analyzer="char", ngram_range=(3, 3))` finds every trigram that occurs
anywhere in the corpus, makes it a column, and counts how often each text contains it. Two
toy strings show the whole mechanism — "yellow banana" and "papagena papaya banana" — and
the columns worth looking at are the ones that occur more than once:


In [ ]:
example = ["yellow banana", "papagena papaya banana"]
toy = CountVectorizer(analyzer="char", ngram_range=(3, 3))
toy_counts = toy.fit_transform(example).toarray()

table = pd.DataFrame(toy_counts, index=pd.Index(example), columns=toy.get_feature_names_out())
print(f"{table.shape[1]} distinct trigrams, for example {list(table.columns[:6])}")
table.loc[:, table.sum() > 1]


Thirteen and twenty-three characters became two rows of 23 counts. `ana` is the column
both strings share — twice each, because "banana" contains it twice — and `apa` and `pap`
are the second string repeating itself in "papagena papaya". Most of each row is zero, since
every trigram in the corpus is a column and any one text contains few of them; that stays
true at scale, where the hundred chunks below span some twenty thousand distinct trigrams.

### Manhattan distance: how different are two rows of counts?

Two rows, one number for how far apart they are. The **Manhattan distance** adds up the
absolute difference per column — `|2 − 2| + |1 − 0| + …` — walking the grid block by block
instead of cutting the diagonal the way Euclidean distance would. On counts it reads
directly: *these two texts differ by 21 trigram occurrences in total*. Identical rows sit at
distance zero. It is also why the chunks had to be equal: a chunk twice as long has counts
roughly twice as big, and Manhattan would put it far from everything for its length alone.
6.3 introduces cosine similarity, which ignores length by construction; here, equal chunks
do the same job.


In [ ]:
manhattan_distances(toy_counts, toy_counts)


Now the real thing: a hundred chunks, every trigram in the corpus as a column, and the
hundred-by-hundred matrix of distances between them.


In [ ]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3))
counts = vectorizer.fit_transform(chunks.text).toarray()
distance = manhattan_distances(counts, counts)

off_diagonal = distance[np.triu_indices(len(distance), k=1)]
print(f"counts: {counts.shape[0]} chunks x {counts.shape[1]:,} trigrams; "
      f"a chunk uses {np.mean(counts > 0):.0%} of them")
print(f"distance: {distance.shape}; nearest pair {off_diagonal.min():,.0f} apart, "
      f"farthest {off_diagonal.max():,.0f}")


## 6.2.4 Reduce, and look

Each row of the distance matrix describes one chunk by its distance to all hundred others —
a hundred-dimensional profile. Two chunks with similar profiles are alike, and PCA of the
profiles is 6.1's flat shadow, two components of it. Colour is the site, which is also the
period.


In [ ]:
coords = PCA(n_components=2).fit_transform(distance)

settings = PlotSettings(
    figsize=(7, 6),
    title="A hundred chunks of Q by trigram counts: the 4chan months sit apart",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=coords, labels=chunks.label.to_numpy(), palette="Set1")


The ten chunks from the 4chan months sit away from the rest; the 8chan and 8kun chunks lie
on top of each other. That is the paper's picture, more or less — and 6.1's rule applies
before believing it: a projection always fills its frame, so get the number that does not
depend on the picture. The silhouette by site, computed on the full distance matrix,
against what the same score gives when the site labels are shuffled:


In [ ]:
def separation(distance: np.ndarray, labels: np.ndarray, seed: int = 0) -> tuple[float, float]:
    '''Silhouette of the labels on a precomputed distance matrix, and the same for shuffled labels.'''
    rng = np.random.default_rng(seed)
    real = silhouette_score(distance, labels, metric="precomputed")
    shuffled = np.mean([silhouette_score(distance, rng.permutation(labels), metric="precomputed")
                        for _ in range(50)])
    return float(real), float(shuffled)


labels = chunks.label.to_numpy()
real, shuffled = separation(distance, labels)
first_months = np.where(labels == "4ch", "4chan", "after the move")
early_real, early_shuffled = separation(distance, first_months)

print(f"silhouette by site:          {real:.3f}   (shuffled labels: {shuffled:.3f})")
print(f"silhouette 4chan vs. after:  {early_real:.3f}   (shuffled labels: {early_shuffled:.3f})")


Real, and small. Shuffled labels score about zero, as they should; the true sites score a
few hundredths above it, and almost all of that is the ten 4chan chunks against everything
else — the 8chan/8kun split adds nothing. So the finding the counts support so far is
narrower than "two people": **the first two months read differently from the rest.**

### Which trigrams do the separating

A count vector can be read back, which is the whole advantage of a hand-designed
representation over a learned one. Per trigram, its share of a chunk on 4chan minus its
share after the move; the extremes are the difference in a list:


In [ ]:
share = counts / counts.sum(axis=1, keepdims=True)
early = labels == "4ch"
difference = pd.Series(share[early].mean(axis=0) - share[~early].mean(axis=0),
                       index=vectorizer.get_feature_names_out())

print("more common in the 4chan months:", [repr(t) for t in difference.nlargest(10).index])
print("more common after the move:     ", [repr(t) for t in difference.nsmallest(10).index])


Two different kinds of thing are on those lists. The 4chan side is **questions** — `wha`,
`why`, `? w`, `is ` — which is a fact about the writing: the early posts were strings of
Socratic questions. The other side is nothing but **links**: `htt`, `://`, `com`, `om/`.
A link is pasted, not typed, and the domain in it names the platform rather than the
writer. A person who moves from 4chan to 8chan and starts replying with links changes their
trigram profile without changing who they are — and a trigram counter cannot tell the two
kinds of change apart.

## 6.2.5 Verify before claiming a mechanism

Lesson 3's rule: check the mechanism you are about to write down. The cheapest alternative
explanation for the picture is "the platform's markup", and it can be removed:
`NormaliseText` takes two more decisions — strip URLs, keep letters and spaces only — so the
second pass is the same pipeline with two parameters changed and everything else identical.
The count-and-distance steps are the ones `wa_analyzer.model.TextClustering` wraps, so from
here they are one call:


In [ ]:
clustering = TextClustering()
assert np.allclose(clustering.fit(chunks.text.tolist()), distance)  # noqa: S101 -- the class is the cells above

stripped = (
    Pipeline()
    .add(NormaliseText, column="text", strip_urls=True, letters_only=True)
    .add(Filter, expr="n_chars > 50")
    .apply(posts)
)
letter_chunks = build_chunks(stripped, size=stripped.n_chars.sum() // N_CHUNKS, label="site")
letter_distance = clustering.fit(letter_chunks.text.tolist())
letter_labels = letter_chunks.label.to_numpy()
letter_early, _ = separation(letter_distance, np.where(letter_labels == "4ch", "4chan", "after the move"))

print(f"silhouette, 4chan vs. after:  everything counted {early_real:.3f}   "
      f"letters and spaces only {letter_early:.3f}")

passes = PlotSettings(
    figsize=(12, 5),
    title="The same posts, counted twice",
    subplot_titles=["everything, links and markup included", "letters and spaces only"],
    xlabel="",
    ylabel="",
    max_cols=2,
)
host = ProjectionPlot(passes)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(ProjectionPlot(passes), axes[0], coordinates=coords, labels=labels,
                  palette="Set1", legend=False)
host.plot_on_axes(ProjectionPlot(passes), axes[1], coordinates=clustering.reduce_dims(letter_distance),
                  labels=letter_labels, palette="Set1")


The 4chan chunks sit apart just as clearly once every link, digit and punctuation mark is
gone — the silhouette does not move. So the split is not the platform's markup: it is in
the words. That rules out the cheapest explanation, and leaves the two the counts cannot
separate. Run it through 05.1's grid:

- **Evidence.** A real structure, small, resting on ten chunks against ninety, and the
  pattern is "the first two months differ", not "two clusters".
- **Mechanism.** Two candidates, and both predict exactly this picture: a second author
  took over, or the same author changed register — from Socratic questions on one board to
  answers and links on another. The switch of author, if there was one, coincides with the
  switch of platform by construction, so nothing in this corpus can pull them apart.
- **Replication.** None available. One corpus, one switch.

**Verdict: plausible, unproven.** What would settle it is text by the same known writer
across the same kind of platform move, and nobody has that. The press release said "two
different people"; the count vectors say "a change in the writing, at the time the platform
changed, and not made of the platform's markup". That second sentence is the one this
notebook can defend.


## 6.2.6 Your turn

Same four steps, your own chat, with one difference you already know how to handle: the
authors are labelled, so the question is not "how many people" but **does typing style
separate the people you know are there?** Per author: normalise, chunk, count, distance —
`ChunkByCharacters(by="author")` restarts the chunk numbering per person — then the
picture and the number.

The chunk size is a decision again. A chat has far less text per person than Q posted, so
the chunks below are 500 characters rather than 8,000, and the counts in them are noisier
for it. Raise it if you have the text, and watch what the silhouette does; lower it and
watch it fall.


In [ ]:
own = load_own_chat().sort_values(["author", "timestamp"])

own_clean = (
    Pipeline()
    .add(NormaliseText, column="message", strip_urls=True)
    .add(Filter, expr="n_chars > 20")
    .apply(own)
)

OWN_CHUNK = 500
own_chunks = build_chunks(own_clean, size=OWN_CHUNK, label="author", by="author")
enough = own_chunks.label.value_counts()
own_chunks = own_chunks[own_chunks.label.isin(enough[enough >= 3].index)].reset_index(drop=True)
print(f"{len(own_chunks)} chunks of {OWN_CHUNK} characters, "
      f"{own_chunks.label.nunique()} authors with three or more")

own_distance = clustering.fit(own_chunks.text.tolist())
own_real, own_shuffled = separation(own_distance, own_chunks.label.to_numpy())
print(f"silhouette by author: {own_real:.3f}   (shuffled labels: {own_shuffled:.3f})")

mine = PlotSettings(
    figsize=(7, 6),
    title=f"{OWN_CHUNK}-character chunks of your chat, by trigram counts",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(mine).plot(coordinates=clustering.reduce_dims(own_distance),
                                    labels=own_chunks.label.to_numpy(), palette="Set1")


## What this baseline is for

Trigram counts separated the 4chan months because the claim is about *style* —
punctuation, spacing, word choice — and three-character sequences carry a lot of that.
Your own chat will usually not split as cleanly, and that is not a failure of the method:
a silhouette near the shuffled one means these people, in chunks this size, do not type
differently enough to be told apart by counting characters. Say so, with the chunk size
and the number of chunks per person in the sentence.

Keep the shape of the pipeline in mind — normalise, chunk, count, distance, look — because
6.3 and 6.4 run the same steps with one change: what gets counted is no longer chosen by
you. And keep the reading habit from §6.2.4: a picture that fills its frame, a number that
does not depend on the picture, and a look at *which* features did the work before a
mechanism gets written down.
